# IMDB Sentiment Analysis Assignment (Week 3.2)

This notebook completes **all required steps** in the prompt:

- **Part 1:** TextBlob sentiment classification + accuracy (and optional VADER extra credit)
- **Part 2:** Text preprocessing (lowercase, remove punctuation/special characters, remove stop words, Porter stemming)
  + Bag-of-Words matrix + TF‑IDF matrix (with dimensions shown)


In [12]:
# -----------------------------
# 0) Setup: imports + (optional) auto-install
# -----------------------------
import sys
import subprocess
from pathlib import Path

def pip_install(package: str) -> None:
    """Install a package into the current Python environment (kernel)."""
    subprocess.check_call([sys.executable, "-m", "pip", "install", package])

# Core packages
import numpy as np
import pandas as pd

# Jupyter display helper
try:
    from IPython.display import display
except Exception:
    # If IPython isn't available, define a safe fallback
    def display(x):
        print(x)

# Ensure TextBlob exists (required for Part 1)
try:
    from textblob import TextBlob
except ModuleNotFoundError:
    print("Installing textblob...")
    pip_install("textblob")
    from textblob import TextBlob

# Ensure NLTK exists (required for stemming; VADER optional)
try:
    import nltk
except ModuleNotFoundError:
    print("Installing nltk...")
    pip_install("nltk")
    import nltk

# Ensure scikit-learn exists (required for BoW and TF-IDF)
try:
    from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer, ENGLISH_STOP_WORDS
    from sklearn.metrics import accuracy_score
except ModuleNotFoundError:
    print("Installing scikit-learn...")
    pip_install("scikit-learn")
    from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer, ENGLISH_STOP_WORDS
    from sklearn.metrics import accuracy_score

print("Environment ready ✅")
print("Python:", sys.executable)


Environment ready ✅
Python: C:\Users\adjov\Downloads\vivi folder\python.exe


## Part 1 — Using the TextBlob Sentiment Analyzer

In [2]:
# -----------------------------
# 1) Load the IMDB labeled training data
# -----------------------------
# The Kaggle file is usually named labeledTrainData.tsv (or .tsv.zip).
# This code finds it in the CURRENT folder and loads it as a DataFrame.

candidate_files = [
    "labeledTrainData.tsv",
    "labeledTrainData.tsv.zip",
    "labeledTrainData.tsv.gz",
]

data_path = None
cwd = Path.cwd()

for fname in candidate_files:
    p = cwd / fname
    if p.exists():
        data_path = p
        break

if data_path is None:
    # If the notebook was opened from a different folder, also try searching one level down.
    found = list(cwd.rglob("labeledTrainData.tsv")) + list(cwd.rglob("labeledTrainData.tsv.zip"))
    if found:
        data_path = found[0]

if data_path is None:
    raise FileNotFoundError(
        "Could not find labeledTrainData.tsv or labeledTrainData.tsv.zip in this folder. "
        "Move the dataset into the same folder as the notebook, then rerun."
    )

print("Loading:", data_path)

# Load TSV (pandas can read .zip directly with compression='infer')
df = pd.read_csv(data_path, sep="\t", compression="infer", quoting=3)

# Quick checks requested by the assignment
print("DataFrame shape:", df.shape)
display(df.head())


Loading: C:\Users\adjov\Downloads\vivi folder\week 3\labeledTrainData.tsv.zip
DataFrame shape: (25000, 3)


,id,sentiment,review
0,"""5814_8""",1,"""With all this stuff going down at the moment ..."
1,"""2381_9""",1,"""\""The Classic War of the Worlds\"" by Timothy ..."
2,"""7759_3""",0,"""The film starts with a manager (Nicholas Bell..."
3,"""3630_4""",0,"""It must be assumed that those who praised thi..."
4,"""9495_8""",1,"""Superbly trashy and wondrously unpretentious ..."


In [3]:
# -----------------------------
# 2) Count how many positive vs. negative reviews are in the dataset
# -----------------------------
# In this dataset: sentiment 0 = negative, 1 = positive

sent_counts = df["sentiment"].value_counts().sort_index()
print("Negative (0):", int(sent_counts.get(0, 0)))
print("Positive (1):", int(sent_counts.get(1, 0)))
display(sent_counts)


Negative (0): 12500
Positive (1): 12500


sentiment
0    12500
1    12500
Name: count, dtype: int64

In [10]:
# -----------------------------
# 3) Use TextBlob to classify each review
# NOTE: Running TextBlob on the FULL 25,000 reviews can take a while on some laptops.
# If you want to test quickly first, set SAMPLE_SIZE to 500 or 2000, then switch back to None for final submission.
# Rule: polarity >= 0 => positive (1); polarity < 0 => negative (0)
# -----------------------------

def textblob_predict(review_text: str) -> tuple[float, int]:
    polarity = TextBlob(review_text).sentiment.polarity
    pred = 1 if polarity >= 0 else 0
    return polarity, pred

# Optional: set to an integer (e.g., 2000) for faster testing; keep None for full dataset.
SAMPLE_SIZE = None

work_df = df.copy()
if SAMPLE_SIZE is not None:
    work_df = work_df.sample(SAMPLE_SIZE, random_state=42).reset_index(drop=True)
    print(f"Using a sample of {SAMPLE_SIZE} rows for speed (set SAMPLE_SIZE=None for full dataset).")

tb_results = work_df["review"].apply(textblob_predict)
work_df["tb_polarity"] = tb_results.apply(lambda x: x[0])
work_df["tb_pred"] = tb_results.apply(lambda x: x[1]).astype(int)

display(work_df[["id", "sentiment", "tb_polarity", "tb_pred"]].head())


,id,sentiment,tb_polarity,tb_pred
0,"""5814_8""",1,0.001277,1
1,"""2381_9""",1,0.256349,1
2,"""7759_3""",0,-0.053941,0
3,"""3630_4""",0,0.134753,1
4,"""9495_8""",1,-0.024290,0


In [5]:
# -----------------------------
# 4) Accuracy of TextBlob classifier + compare to random guessing
# -----------------------------
y_true = work_df["sentiment"].astype(int)
y_pred_tb = work_df["tb_pred"].astype(int)

acc_tb = accuracy_score(y_true, y_pred_tb)
print(f"TextBlob accuracy: {acc_tb:.4f} ({acc_tb*100:.2f}%)")

random_guess_baseline = 0.50
print(f"Baseline (random guessing, expected): {random_guess_baseline:.2f}")
print("Conclusion:", "Better than random guessing ✅" if acc_tb > random_guess_baseline else "NOT better than random ❌")


TextBlob accuracy: 0.6850 (68.50%)
Baseline (random guessing, expected): 0.50
Conclusion: Better than random guessing ✅


## Extra Credit (Optional) — VADER Sentiment Analyzer

In [6]:
# -----------------------------
# Extra Credit: VADER sentiment classifier (compound >= 0 => positive, < 0 => negative)
# Repeats: classification rule + accuracy + better-than-random conclusion
# -----------------------------
try:
    from nltk.sentiment import SentimentIntensityAnalyzer

    # Create analyzer (download lexicon if missing)
    try:
        sia = SentimentIntensityAnalyzer()
    except LookupError:
        print("VADER lexicon missing — downloading...")
        nltk.download("vader_lexicon")
        sia = SentimentIntensityAnalyzer()

    work_df["vader_compound"] = work_df["review"].apply(lambda t: sia.polarity_scores(t)["compound"])
    work_df["vader_pred"] = (work_df["vader_compound"] >= 0).astype(int)

    acc_vader = accuracy_score(y_true, work_df["vader_pred"])
    print(f"VADER accuracy: {acc_vader:.4f} ({acc_vader*100:.2f}%)")

    random_guess_baseline = 0.50
    print(f"Baseline (random guessing, expected): {random_guess_baseline:.2f}")
    print("Conclusion:", "Better than random guessing ✅" if acc_vader > random_guess_baseline else "NOT better than random ❌")

    display(work_df[["sentiment", "vader_compound", "vader_pred"]].head())

except Exception as e:
    print("VADER could not be run in this environment.")
    print("Error:", type(e).__name__, "-", str(e)[:200])


VADER accuracy: 0.6924 (69.24%)
Baseline (random guessing, expected): 0.50
Conclusion: Better than random guessing ✅


,sentiment,vader_compound,vader_pred
0,1,-0.8278,0
1,1,0.9819,1
2,0,-0.9883,0
3,0,-0.2189,0
4,1,0.7960,1


## Part 2 — Prepping Text for a Custom Model

In [11]:
# -----------------------------
# 1–4) Preprocess text: lowercase, remove punctuation/special chars, remove stop words, PorterStemmer
# -----------------------------
import re
from nltk.stem import PorterStemmer

stop_words = set(ENGLISH_STOP_WORDS)  # scikit-learn built-in stop word list (no download needed)
stemmer = PorterStemmer()

def preprocess_and_stem(text: str) -> str:
    # 1) Lowercase
    text = text.lower()

    # Remove HTML tags (dataset contains <br /> etc.)
    text = re.sub(r"<[^>]+>", " ", text)

    # 2) Remove punctuation/special characters (keep letters + whitespace)
    text = re.sub(r"[^a-z\s]", " ", text)

    # Normalize spaces
    text = re.sub(r"\s+", " ", text).strip()

    # 3) Remove stop words
    tokens = [w for w in text.split() if w and (w not in stop_words)]

    # 4) Porter stemming
    stems = [stemmer.stem(w) for w in tokens]
    return " ".join(stems)

work_df["stemmed_review"] = work_df["review"].astype(str).apply(preprocess_and_stem)

print("Example (original vs stemmed):")
print("\nORIGINAL:\n", work_df.loc[0, "review"][:300], "...")
print("\nSTEMMED:\n", work_df.loc[0, "stemmed_review"][:300], "...")


Example (original vs stemmed):

ORIGINAL:
 "With all this stuff going down at the moment with MJ i've started listening to his music, watching the odd documentary here and there, watched The Wiz and watched Moonwalker again. Maybe i just want to get a certain insight into this guy who i thought was really cool in the eighties just to maybe m ...

STEMMED:
 stuff go moment mj ve start listen music watch odd documentari watch wiz watch moonwalk mayb just want certain insight guy thought realli cool eighti just mayb make mind guilti innoc moonwalk biographi featur film rememb go cinema origin releas subtl messag mj s feel press obviou messag drug bad m k ...


In [8]:
# -----------------------------
# 5) Bag-of-Words matrix + dimensions
# Rows must match DataFrame rows
# -----------------------------
bow_vectorizer = CountVectorizer(lowercase=False)
X_bow = bow_vectorizer.fit_transform(work_df["stemmed_review"])

print("BoW matrix type:", type(X_bow))
print("BoW dimensions (rows, cols):", X_bow.shape)
print("DataFrame rows:", work_df.shape[0])

assert X_bow.shape[0] == work_df.shape[0]
print("✅ BoW row count matches DataFrame row count")


BoW matrix type: <class 'scipy.sparse._csr.csr_matrix'>
BoW dimensions (rows, cols): (25000, 49566)
DataFrame rows: 25000
✅ BoW row count matches DataFrame row count


In [9]:
# -----------------------------
# 6) TF-IDF matrix + dimensions (must match BoW dimensions)
# -----------------------------
tfidf_vectorizer = TfidfVectorizer(lowercase=False)
X_tfidf = tfidf_vectorizer.fit_transform(work_df["stemmed_review"])

print("TF-IDF matrix type:", type(X_tfidf))
print("TF-IDF dimensions (rows, cols):", X_tfidf.shape)

assert X_tfidf.shape == X_bow.shape
print("✅ TF-IDF dimensions match BoW dimensions")


TF-IDF matrix type: <class 'scipy.sparse._csr.csr_matrix'>
TF-IDF dimensions (rows, cols): (25000, 49566)
✅ TF-IDF dimensions match BoW dimensions
